# 03 — Prepare

Two outputs, both from `games_clean`:

1. **Sellable export** — `export/video_game_scores_v1` as CSV + Excel + Parquet, with a plain-English **codebook** (every column described). No PII (public game metadata), so `strip_pii` is a no-op.
2. **`chart_*` DuckDB tables** for the confirmed charts:
   - `chart_critic_all` — one row per game (critic + user rating) → the two distribution histograms (critic and user).
   - `chart_90plus_by_year` — count of 90+ critic games per release year, **cut at the last complete year** (recent years still accruing critic scores). (Exploration reference; not in the published social set.)
   - `chart_avg_by_year` — mean critic vs mean user rating per year → the two-line chart.
   - `chart_scatter` — games with BOTH ratings, `critic_rating` > 0, with an `is_outlier` flag (12 most visually-isolated points) → the user-vs-critic scatter.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import get_connection
from src.prepare import package_dataset

cfg = load_config('config.yaml')
con = get_connection(cfg)
print('games_clean rows:', con.execute('SELECT COUNT(*) FROM games_clean').fetchone()[0])

## 1. Sellable export + codebook

In [ ]:
export_df = con.execute('''
    SELECT id, name, slug, critic_rating, critic_count,
           user_rating, user_count, release_year, decade,
           genres, platforms
    FROM games_clean
    ORDER BY critic_rating DESC, name
''').df()

codebook = {
    'id': 'IGDB internal game id (unique per game).',
    'name': 'Game title as listed on IGDB.',
    'slug': 'IGDB URL slug for the game.',
    'critic_rating': 'IGDB aggregated CRITIC rating, 0-100 (unweighted mean of external critic outlet scores IGDB has collected). The distribution this project charts.',
    'critic_count': 'Number of external critic scores behind critic_rating (>= 3 by construction).',
    'user_rating': 'IGDB community/USER rating, 0-100 (a separate measure from the critic rating; not blended with it).',
    'user_count': 'Number of IGDB users behind user_rating (may be null if no user ratings).',
    'release_year': 'Year of first release, derived from the IGDB first_release_date timestamp.',
    'decade': 'Release decade (release_year floored to the nearest 10) — for era comparisons.',
    'genres': 'Comma-separated IGDB genres for the game.',
    'platforms': 'Comma-separated platforms the game released on (per IGDB).',
}

notes = '''Source: IGDB — Internet Game Database (API), https://api.igdb.com/v4/games
License: Free for non-commercial use under the Twitch Developer Services Agreement; attribution to IGDB. Fun-tier pop-culture dataset.
Scope: games with a critic aggregate backed by >= 3 critic outlets (aggregated_rating_count >= 3).
Metric: critic_rating is IGDB\'s aggregate of external CRITIC scores (NOT Metacritic; a different aggregator/method). user_rating is IGDB\'s community rating. Two separate measures, never blended.
Caveat: IGDB aggregate is an unweighted mean of whatever outlets it has, so it varies by title/era; scores drift up over time; very recent titles may still be accruing critic scores.'''

written = package_dataset(export_df, cfg, name='video_game_scores_v1', codebook=codebook, notes=notes)
written

## 2a. `chart_critic_all` — the histogram + scatter source
One row per game with both ratings. The histogram bins `critic_rating`; the scatter uses games that also have a `user_rating`.

In [ ]:
con.execute('DROP TABLE IF EXISTS chart_critic_all')
con.execute('''CREATE TABLE chart_critic_all AS
  SELECT id, name, critic_rating, critic_count, user_rating, user_count, release_year, decade
  FROM games_clean''')
print('chart_critic_all:', con.execute('SELECT COUNT(*) FROM chart_critic_all').fetchone()[0])
print('with BOTH critic+user (scatter):', con.execute('SELECT COUNT(*) FROM chart_critic_all WHERE user_rating IS NOT NULL').fetchone()[0])

## 2b. `chart_90plus_by_year` — 90+ critic games per year

Count of games with `critic_rating >= 90` by release year. We **cut at the last year whose catalogue looks complete** — determined data-drivenly below, since very recent releases are still accruing critic scores (a partial year would look like a false collapse). The cutoff year + reason go in the chart caption.

In [ ]:
import datetime
this_year = datetime.date.today().year

# Total scored games per year — find where recent years fall off a cliff.
per_year = con.execute('''
  SELECT release_year,
         COUNT(*) AS total_scored,
         SUM(CASE WHEN critic_rating >= 90 THEN 1 ELSE 0 END) AS n_90plus
  FROM chart_critic_all WHERE release_year IS NOT NULL
  GROUP BY release_year ORDER BY release_year
''').df()

# Cutoff: the last year whose total_scored is at least 40% of the recent peak
# (a simple, defensible completeness rule). Never include the current year.
recent = per_year[per_year.release_year >= this_year - 12]
peak = recent['total_scored'].max()
complete = per_year[(per_year.total_scored >= 0.4 * peak) & (per_year.release_year < this_year)]
CUTOFF_YEAR = int(complete['release_year'].max())
print('recent peak scored/yr:', int(peak), '| chosen cutoff year:', CUTOFF_YEAR)
per_year.tail(12)

In [ ]:
# Start the line where annual counts stabilize (>= 10 scored games/yr), through the cutoff.
start_row = per_year[per_year.total_scored >= 10]
START_YEAR = int(start_row['release_year'].min())
print('line spans', START_YEAR, '→', CUTOFF_YEAR)

con.execute('DROP TABLE IF EXISTS chart_90plus_by_year')
con.execute(f'''CREATE TABLE chart_90plus_by_year AS
  SELECT release_year AS year,
         SUM(CASE WHEN critic_rating >= 90 THEN 1 ELSE 0 END) AS n_90plus
  FROM chart_critic_all
  WHERE release_year BETWEEN {START_YEAR} AND {CUTOFF_YEAR}
  GROUP BY release_year ORDER BY release_year''')
con.execute('SELECT * FROM chart_90plus_by_year').df()

## 2c. `chart_avg_by_year` — average critic vs user rating per year

Mean critic rating and mean user rating for each release year, for the two-line chart (one line critic, one line user). Restricted to years with **≥ 10 scored games** (so early sparse years don't wobble) and cut at the **same last-complete year** as `chart_90plus_by_year` (recent partial years would distort the tail). This is the exact series the `06-viz-social` line chart reads.

In [ ]:
con.execute('DROP TABLE IF EXISTS chart_avg_by_year')
con.execute(f'''CREATE TABLE chart_avg_by_year AS
  SELECT release_year AS year,
         ROUND(AVG(critic_rating), 1) AS avg_critic,
         ROUND(AVG(user_rating), 1)   AS avg_user,
         COUNT(*)                     AS n_games
  FROM chart_critic_all
  WHERE release_year IS NOT NULL
  GROUP BY release_year
  HAVING COUNT(*) >= 10 AND release_year <= {CUTOFF_YEAR}
  ORDER BY release_year''')
con.execute('SELECT * FROM chart_avg_by_year').df()

## 2d. `chart_scatter` — user vs critic, with visual-isolation outlier flag

One row per game that has **both** a critic and user rating, for the scatter. Two prep decisions baked in here (so they live in the pipeline, not just the exploration notebook):

1. **Drop the `critic_rating = 0` artifact** — a single game (Infernal) carries an aggregate of exactly 0 over 6 outlets, which is an IGDB data gap, not a real mean. It would otherwise be the most-isolated dot on the plot.
2. **Flag the 12 most VISUALLY ISOLATED points** as outliers (`is_outlier`) — the games with the most empty space around them, measured as the mean distance to each point's **8 nearest neighbours** in normalized (0–1 per axis) score space. This is what the eye reads as ‘standing out’; it flags the lonely dots in every corner and does not over-pick the dense clusters. These are the games the scatter labels.

In [ ]:
import numpy as np
N_OUTLIERS = 12
K = 8

sc = con.execute('''
  SELECT name, critic_rating, user_rating
  FROM chart_critic_all
  WHERE user_rating IS NOT NULL AND critic_rating > 0
''').df()

cx = sc['critic_rating'].values.astype(float)
uy = sc['user_rating'].values.astype(float)
nx = (cx - cx.min()) / (cx.max() - cx.min())
ny = (uy - uy.min()) / (uy.max() - uy.min())
P = np.column_stack([nx, ny])
iso = np.empty(len(P))
for i in range(0, len(P), 500):                # chunked brute-force kNN (no scipy)
    blk = P[i:i+500]
    d2 = ((blk[:, None, :] - P[None, :, :]) ** 2).sum(2)
    d2.sort(axis=1)
    iso[i:i+500] = np.sqrt(d2[:, 1:K+1]).mean(1)
sc['isolation'] = iso
sc['is_outlier'] = sc.index.isin(sc['isolation'].nlargest(N_OUTLIERS).index)

con.execute('DROP TABLE IF EXISTS chart_scatter')
con.register('sc_df', sc)
con.execute('CREATE TABLE chart_scatter AS SELECT name, critic_rating, user_rating, isolation, is_outlier FROM sc_df')
con.unregister('sc_df')
print('chart_scatter:', con.execute('SELECT COUNT(*) FROM chart_scatter').fetchone()[0], 'games;',
      con.execute('SELECT COUNT(*) FROM chart_scatter WHERE is_outlier').fetchone()[0], 'flagged outliers')
con.execute('SELECT name, critic_rating, user_rating, ROUND(isolation,3) AS isolation FROM chart_scatter WHERE is_outlier ORDER BY isolation DESC').df()

### Sanity: headline distribution facts
The numbers a caption/validation would cite.

In [ ]:
con.execute('''
SELECT COUNT(*) AS n_games,
  ROUND(AVG(critic_rating),1) AS mean_critic,
  ROUND(MEDIAN(critic_rating),1) AS median_critic,
  SUM(CASE WHEN critic_rating >= 90 THEN 1 ELSE 0 END) AS n_90plus,
  SUM(CASE WHEN critic_rating BETWEEN 70 AND 79.999 THEN 1 ELSE 0 END) AS n_70s,
  SUM(CASE WHEN critic_rating < 50 THEN 1 ELSE 0 END) AS n_sub50
FROM chart_critic_all
''').df()

---
**Next:** `04-viz.ipynb` — explore the distribution, the 90+ trend, and the critic-vs-user scatter. **Pause for owner review before `06-viz-social`.**

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')